In [1]:
import glob
import numpy as np
import pandas as pd
from netCDF4 import Dataset
from wrf import getvar, interplevel
from scipy.interpolate import RegularGridInterpolator

# --- Configuration ---
base_path = '/data/nobackup/boland/'

experiments = [
    ('exp1994_WF', 'Current Farms - Current Climate'),
    ('exp1994_WFF', 'Future Farms - Current Climate'),
    ('exp2071_WF', 'Current Farms - Future Climate'),
    ('exp2071_WFF', 'Future Farms - Future Climate')
]

file_pattern = 'WRF/run/wrfout_d03_*-08-*'

# Bounding boxes brought back ONLY to classify the raw nameless coordinates
BELGIAN_INDIVIDUAL_FARMS = {
    "C-Power": {
        "bbox": [51.58, 2.90, 51.53, 3.03], 
        "capacity_gw": 0.325
    },
    "Belwind": {
        "bbox": [51.72, 2.78, 51.65, 2.84], 
        "capacity_gw": 0.165
    },
    "Northwind": {
        "bbox": [51.63, 2.86, 51.58, 2.93], 
        "capacity_gw": 0.216
    },
    "Nobelwind": {
        "bbox": [51.73, 2.82, 51.68, 2.87], 
        "capacity_gw": 0.165
    },
    "Rentel": {
        "bbox": [51.62, 2.76, 51.56, 2.84], 
        "capacity_gw": 0.309
    },
    "Norther": {
        "bbox": [51.55, 3.00, 51.49, 3.08], 
        "capacity_gw": 0.370
    },
    "SeaMade-Seastar": {
        "bbox": [51.65, 2.71, 51.60, 2.77], 
        "capacity_gw": 0.252
    },
    "SeaMade-Mermaid": {
        "bbox": [51.74, 2.72, 51.69, 2.77], 
        "capacity_gw": 0.260
    },
    "PEZ-Noordhinder Noord": {
        "bbox": [51.54, 2.44, 51.46, 2.58], 
        "capacity_gw": 0.700
    },
    "PEZ-Noordhinder Zuid": {
        "bbox": [51.47, 2.46, 51.37, 2.62], 
        "capacity_gw": 1.400
    },
    "PEZ-Fairy Bank": {
        "bbox": [51.44, 2.63, 51.36, 2.76], 
        "capacity_gw": 1.400
    }
}

def wind_speed_to_capacity_factor(wspd):
    wspd_nodes = np.array([0.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 11.5, 25.0, 99.0])
    power_nodes = np.array([0.0, 0.0, 0.53, 1.17, 2.15, 3.54, 5.43, 7.91, 11.07, 14.12, 15.00, 15.00, 0.0])
    computed_power_mw = np.interp(wspd, wspd_nodes, power_nodes)
    cf = computed_power_mw / 15.0
    cf = np.where(wspd > 25.0, 0.0, cf)
    return np.clip(cf, 0.0, 1.0)

# --- Geographic Coordinate Sorter ---
def load_and_sort_turbines(filepath):
    turbines_by_farm = {farm: [] for farm in BELGIAN_INDIVIDUAL_FARMS.keys()}
    unassigned_count = 0
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            
            parts = line.split()
            if len(parts) >= 2:
                try:
                    lat = float(parts[0])
                    lon = float(parts[1])
                    
                    # Check which farm bounding box this turbine falls into
                    assigned = False
                    for farm_name, info in BELGIAN_INDIVIDUAL_FARMS.items():
                        north, west, south, east = info["bbox"]
                        
                        # Match coordinate to bounding box
                        if (lat >= south) and (lat <= north) and (lon >= west) and (lon <= east):
                            turbines_by_farm[farm_name].append((lat, lon))
                            assigned = True
                            break
                    
                    if not assigned:
                        unassigned_count += 1
                except ValueError:
                    continue
                    
    if unassigned_count > 0:
        print(f"Notice: {unassigned_count} turbines in file fell outside all defined Belgian farm boxes.")
    return turbines_by_farm

# Load and sort coordinates using the bounding box geography checker
# turbine_layout = load_and_sort_turbines('windturbines.txt')

# Dictionary for raw total energy values
raw_energy = {farm: {} for farm in BELGIAN_INDIVIDUAL_FARMS.keys()}
raw_energy["NATIONAL TOTAL (GWh)"] = {}

# --- Processing Loop ---
for folder, exp_name in experiments:
    
    current_turbine_file = f"{base_path}{folder}/WRF/run/windturbines.txt" 
    
    print(f"\nLoading turbine layout from: {current_turbine_file}")
    turbine_layout = load_and_sort_turbines(current_turbine_file)
    
    files = sorted(glob.glob(f"{base_path}{folder}/{file_pattern}"))
    files = [f for f in files if int(f.split('_')[-2].split('-')[-1]) <= 31]
    
    if not files:
        print(f"Warning: No files found for {exp_name}")
        continue
    
    print(f"\nStarting extraction for: {exp_name} ({len(files)} files found)")
    
    total_cf_accumulated = None
    total_hours = 0  

    for i, f in enumerate(files, 1):
        print(f"   [{i}/{len(files)}] Processing file: {f.split('/')[-1]}", end='\r')
        
        nc = Dataset(f)
        u = getvar(nc, "ua", timeidx=None, units="m s-1", meta=False)
        v = getvar(nc, "va", timeidx=None, units="m s-1", meta=False)
        height = getvar(nc, "height_agl", timeidx=None, units="m", meta=False)
        
        wspd = np.sqrt(u**2 + v**2)
        wspd_150m = interplevel(wspd, height, 150.0, meta=False)
        cf_snapshots = wind_speed_to_capacity_factor(wspd_150m)
        
        if cf_snapshots.ndim == 3:  
            hours_in_this_file = cf_snapshots.shape[0]
            cf_day_sum = np.sum(cf_snapshots, axis=0)
        else:  
            hours_in_this_file = 1
            cf_day_sum = cf_snapshots
            
        if total_cf_accumulated is None:
            total_cf_accumulated = cf_day_sum
            lats = getvar(nc, "XLAT", timeidx=0, meta=False)
            lons = getvar(nc, "XLONG", timeidx=0, meta=False)
        else:
            total_cf_accumulated += cf_day_sum
            
        total_hours += hours_in_this_file
        nc.close()
        
    print(f"\n   Finished combining grids for {exp_name}.")
    mean_cf_grid = total_cf_accumulated / total_hours
    
    # --- Spatial Interpolation Setup ---
    lat_1d = lats[:, 0]
    lon_1d = lons[0, :]
    interp_cf = RegularGridInterpolator((lat_1d, lon_1d), mean_cf_grid, method='linear', bounds_error=False, fill_value=0.0)
    
    national_sum_gwh = 0.0

    for farm_name, info in BELGIAN_INDIVIDUAL_FARMS.items():
        capacity_gw = info["capacity_gw"]
        
        # DYNAMIC FILTER: Skip future PEZ zones if analyzing "Current Farms" scenarios
        if "Current Farms" in exp_name and farm_name.startswith("PEZ-"):
            raw_energy[farm_name][exp_name] = 0.0
            continue
            
        coords = turbine_layout.get(farm_name, [])
        
        if len(coords) > 0:
            # Extracts precise capacity factors at exact wake-affected coordinates
            turbine_cfs = interp_cf(coords)
            farm_mean_cf = np.mean(turbine_cfs)
            farm_energy_gwh = farm_mean_cf * capacity_gw * total_hours
            
            raw_energy[farm_name][exp_name] = farm_energy_gwh
            national_sum_gwh += farm_energy_gwh
        else:
            raw_energy[farm_name][exp_name] = 0.0
            
    raw_energy["NATIONAL TOTAL (GWh)"][exp_name] = national_sum_gwh

# --- Convert to DataFrames for easy vector math ---
df_energy = pd.DataFrame(raw_energy).T

# --- Calculate Percentage Differences ---
if 'Future Farms - Current Climate' in df_energy.columns and 'Current Farms - Current Climate' in df_energy.columns:
    df_energy['Expansion Impact (Current Climate) %'] = ((df_energy['Future Farms - Current Climate'] - df_energy['Current Farms - Current Climate']) / df_energy['Current Farms - Current Climate']) * 100

if 'Future Farms - Future Climate' in df_energy.columns and 'Current Farms - Future Climate' in df_energy.columns:
    df_energy['Expansion Impact (Future Climate) %'] = ((df_energy['Future Farms - Future Climate'] - df_energy['Current Farms - Future Climate']) / df_energy['Current Farms - Future Climate']) * 100

if 'Current Farms - Future Climate' in df_energy.columns and 'Current Farms - Current Climate' in df_energy.columns:
    df_energy['Climate Delta (Current Farms) %'] = ((df_energy['Current Farms - Future Climate'] - df_energy['Current Farms - Current Climate']) / df_energy['Current Farms - Current Climate']) * 100

if 'Future Farms - Future Climate' in df_energy.columns and 'Future Farms - Current Climate' in df_energy.columns:
    df_energy['Climate Delta (Future Farms) %'] = ((df_energy['Future Farms - Future Climate'] - df_energy['Future Farms - Current Climate']) / df_energy['Future Farms - Current Climate']) * 100

# --- Output Final Combined Table ---
print("\n" + "="*45 + " COUPLING ANALYSIS: ENERGY & IMPACT PERCENTAGES " + "="*45)
print(df_energy.round(2).to_markdown())


Loading turbine layout from: /data/nobackup/boland/exp1994_WF/WRF/run/windturbines.txt
Notice: 1473 turbines in file fell outside all defined Belgian farm boxes.

Starting extraction for: Current Farms - Current Climate (31 files found)
   [31/31] Processing file: wrfout_d03_1994-08-31_00:00:00
   Finished combining grids for Current Farms - Current Climate.

Loading turbine layout from: /data/nobackup/boland/exp1994_WFF/WRF/run/windturbines.txt
Notice: 7444 turbines in file fell outside all defined Belgian farm boxes.

Starting extraction for: Future Farms - Current Climate (31 files found)
   [31/31] Processing file: wrfout_d03_1994-08-31_00:00:00
   Finished combining grids for Future Farms - Current Climate.

Loading turbine layout from: /data/nobackup/boland/exp2071_WF/WRF/run/windturbines.txt
Notice: 1473 turbines in file fell outside all defined Belgian farm boxes.

Starting extraction for: Current Farms - Future Climate (31 files found)
   [31/31] Processing file: wrfout_d03_2